In [40]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing.transactionencoder import TransactionEncoder
import csv
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [41]:
# 读取原始数据
movies = pd.read_excel('TOP250.xlsx')
movies.head(10)  

,片名,上映年份,评分,评价人数,导演,编剧,主演,类型,国家/地区,语言,时长(分钟)
0,肖申克的救赎,1994,9.7,2317937,弗兰克·德拉邦特,弗兰克·德拉邦特 / 斯蒂芬·金,蒂姆·罗宾斯 / 摩根·弗里曼 / 鲍勃·冈顿 / 威廉姆·赛德勒 / 克兰西·布朗 / 吉...,剧情 / 犯罪,美国,英语,142
1,霸王别姬,1993,9.6,1720638,陈凯歌,芦苇 / 李碧华,张国荣 / 张丰毅 / 巩俐 / 葛优 / 英达 / 蒋雯丽 / 吴大维 / 吕齐 / 雷汉...,剧情 / 爱情 / 同性,中国,汉语普通话,171
2,阿甘正传,1994,9.5,1743966,罗伯特·泽米吉斯,艾瑞克·罗斯 / 温斯顿·格鲁姆,汤姆·汉克斯 / 罗宾·怀特 / 加里·西尼斯 / 麦凯尔泰·威廉逊 / 莎莉·菲尔德 / ...,剧情 / 爱情,美国,英语,142
3,这个杀手不太冷,1994,9.4,1922740,吕克·贝松,吕克·贝松,让·雷诺 / 娜塔莉·波特曼 / 加里·奥德曼 / 丹尼·爱罗 / 彼得·阿佩尔 / 迈克尔...,剧情 / 动作 / 犯罪,法国,英语,110
4,泰坦尼克号,1997,9.4,1706127,詹姆斯·卡梅隆,詹姆斯·卡梅隆,莱昂纳多·迪卡普里奥 / 凯特·温丝莱特 / 比利·赞恩 / 凯西·贝茨 / 弗兰西丝·费舍...,剧情 / 爱情 / 灾难,美国,英语,194
5,美丽人生,1997,9.5,1075345,罗伯托·贝尼尼,温琴佐·切拉米 / 罗伯托·贝尼尼,罗伯托·贝尼尼 / 尼可莱塔·布拉斯基 / 乔治·坎塔里尼 / 朱斯蒂诺·杜拉诺 / 赛尔乔...,剧情 / 喜剧 / 爱情 / 战争,意大利,意大利语,116
6,千与千寻,2001,9.4,1822369,宫崎骏,宫崎骏,柊瑠美 / 入野自由 / 夏木真理 / 菅原文太 / 中村彰男 / 玉井夕海 / 神木隆之介...,剧情 / 动画 / 奇幻,日本,日语,125
7,辛德勒的名单,1993,9.5,890468,史蒂文·斯皮尔伯格,托马斯·肯尼利 / 斯蒂文·泽里安,连姆·尼森 / 本·金斯利 / 拉尔夫·费因斯 / 卡罗琳·古多尔 / 乔纳森·萨加尔 / ...,剧情 / 历史 / 战争,美国,英语,195
8,盗梦空间,2010,9.3,1688052,克里斯托弗·诺兰,克里斯托弗·诺兰,莱昂纳多·迪卡普里奥 / 约瑟夫·高登-莱维特 / 艾利奥特·佩吉 / 汤姆·哈迪 / 渡边...,剧情 / 科幻 / 悬疑 / 冒险,美国,英语,148
9,忠犬八公的故事,2009,9.4,1158337,拉斯·霍尔斯道姆,斯蒂芬·P·林赛 / 新藤兼人,理查·基尔 / 萨拉·罗默尔 / 琼·艾伦 / 罗比·萨布莱特 / 艾瑞克·阿瓦利 / 田川...,剧情,美国,英语,93


# 1对类型进行分析

In [42]:
mov=movies[['片名','类型']]
mov.head()

,片名,类型
0,肖申克的救赎,剧情 / 犯罪
1,霸王别姬,剧情 / 爱情 / 同性
2,阿甘正传,剧情 / 爱情
3,这个杀手不太冷,剧情 / 动作 / 犯罪
4,泰坦尼克号,剧情 / 爱情 / 灾难


In [43]:
# 第一步，将原始数据转化为标准格式
movies_standard = mov.drop('类型', axis=1).join(mov['类型'].replace(r'\s*/\s*', '|', regex=True).str.strip().str.get_dummies())
'''
如果是|隔开，直接用
movies_standard = mov.drop('类型', 1).join(mov['类型'].str.get_dummies())

'''

"\n如果是|隔开，直接用\nmovies_standard = movies.drop('genres', 1).join(movies.genres.str.get_dummies())\n\n"

In [44]:

print(movies_standard.head(10))
# 一共包含 250部电影，一共有29种不同的电影类型（有2列是ID和电影名）
print(movies_standard.shape)  # (250, 30)

        片名  life-is-fruity.com  传记  儿童  冒险  剧情  动作  动画  历史  古装  ...  武侠  \
0   肖申克的救赎                   0   0   0   0   1   0   0   0   0  ...   0   
1     霸王别姬                   0   0   0   0   1   0   0   0   0  ...   0   
2     阿甘正传                   0   0   0   0   1   0   0   0   0  ...   0   
3  这个杀手不太冷                   0   0   0   0   1   1   0   0   0  ...   0   
4    泰坦尼克号                   0   0   0   0   1   0   0   0   0  ...   0   
5     美丽人生                   0   0   0   0   1   0   0   0   0  ...   0   
6     千与千寻                   0   0   0   0   1   0   1   0   0  ...   0   
7   辛德勒的名单                   0   0   0   0   1   0   0   1   0  ...   0   
8     盗梦空间                   0   0   0   1   1   0   0   0   0  ...   0   
9  忠犬八公的故事                   0   0   0   0   1   0   0   0   0  ...   0   

   汉语普通话  灾难  爱情  犯罪  科幻  纪录片  西部  运动  音乐  
0      0   0   0   1   0    0   0   0   0  
1      0   0   1   0   0    0   0   0   0  
2      0   0   1   0   0    0   0   0   0 

In [45]:
movies_standard.columns

Index(['片名', 'life-is-fruity.com', '传记', '儿童', '冒险', '剧情', '动作', '动画', '历史',
       '古装', '同性', '喜剧', '奇幻', '家庭', '恐怖', '悬疑', '情色', '惊悚', '战争', '歌舞', '武侠',
       '汉语普通话', '灾难', '爱情', '犯罪', '科幻', '纪录片', '西部', '运动', '音乐'],
      dtype='object')

In [46]:
# 利用mlxtend提供的apriori算法函数得到频繁项集，其中设置最小支持度为0.05
movies_standard.set_index(['片名'], inplace=True)
frequent_item_sets = apriori(movies_standard, min_support=0.05, use_colnames=True)
print(frequent_item_sets)

    support      itemsets
0     0.060          (传记)
1     0.180          (冒险)
2     0.736          (剧情)
3     0.132          (动作)
4     0.144          (动画)
5     0.224          (喜剧)
6     0.164          (奇幻)
7     0.084          (家庭)
8     0.128          (悬疑)
9     0.132          (惊悚)
10    0.064          (战争)
11    0.220          (爱情)
12    0.172          (犯罪)
13    0.088          (科幻)
14    0.060      (剧情, 传记)
15    0.056      (冒险, 剧情)
16    0.088      (冒险, 动画)
17    0.096      (冒险, 奇幻)
18    0.068      (动作, 剧情)
19    0.144      (剧情, 喜剧)
20    0.084      (家庭, 剧情)
21    0.088      (悬疑, 剧情)
22    0.084      (惊悚, 剧情)
23    0.056      (战争, 剧情)
24    0.188      (剧情, 爱情)
25    0.160      (剧情, 犯罪)
26    0.052      (动画, 喜剧)
27    0.076      (奇幻, 动画)
28    0.072      (喜剧, 爱情)
29    0.088      (惊悚, 悬疑)
30    0.052      (悬疑, 犯罪)
31    0.052  (冒险, 奇幻, 动画)
32    0.052  (惊悚, 悬疑, 剧情)


D:\soft\python\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:113: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  DeprecationWarning,


In [47]:
# 计算规则，并设置提升度阈值为 1.25 （返回的是各个指标的数值，可以按照按兴趣的指标排序观察，但具体解释还得参考实际数据的含义）
rules = association_rules(frequent_item_sets, metric='lift', min_threshold=1.25)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(剧情),(传记),0.736,0.060,0.060,0.081522,1.358696,0.015840,1.023432,1.000000
1,(传记),(剧情),0.060,0.736,0.060,1.000000,1.358696,0.015840,inf,0.280851
2,(冒险),(动画),0.180,0.144,0.088,0.488889,3.395062,0.062080,1.674783,0.860310
3,(动画),(冒险),0.144,0.180,0.088,0.611111,3.395062,0.062080,2.108571,0.824129
4,(冒险),(奇幻),0.180,0.164,0.096,0.533333,3.252033,0.066480,1.791429,0.844512
5,(奇幻),(冒险),0.164,0.180,0.096,0.585366,3.252033,0.066480,1.977647,0.828349
6,(家庭),(剧情),0.084,0.736,0.084,1.000000,1.358696,0.022176,inf,0.288210
7,(剧情),(家庭),0.736,0.084,0.084,0.114130,1.358696,0.022176,1.034012,1.000000
8,(剧情),(犯罪),0.736,0.172,0.160,0.217391,1.263903,0.033408,1.058000,0.790909
9,(犯罪),(剧情),0.172,0.736,0.160,0.930233,1.263903,0.033408,3.784000,0.252174


In [48]:
# 对lift降序排序，查看lift较大的是哪些规则
rules_sort = rules.sort_values(by=['lift'], ascending=False)
rules_sort

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
17,(悬疑),(惊悚),0.128,0.132,0.088,0.687500,5.208333,0.071104,2.777600,0.926606
16,(惊悚),(悬疑),0.132,0.128,0.088,0.666667,5.208333,0.071104,2.616000,0.930876
29,(悬疑),"(惊悚, 剧情)",0.128,0.084,0.052,0.406250,4.836310,0.041248,1.542737,0.909668
26,"(惊悚, 剧情)",(悬疑),0.084,0.128,0.052,0.619048,4.836310,0.041248,2.289000,0.865972
28,(惊悚),"(悬疑, 剧情)",0.132,0.088,0.052,0.393939,4.476584,0.040384,1.504800,0.894718
27,"(悬疑, 剧情)",(惊悚),0.088,0.132,0.052,0.590909,4.476584,0.040384,2.121778,0.851552
22,"(奇幻, 动画)",(冒险),0.076,0.180,0.052,0.684211,3.801170,0.038320,2.596667,0.797536
23,(冒险),"(奇幻, 动画)",0.180,0.076,0.052,0.288889,3.801170,0.038320,1.299375,0.898687
20,"(冒险, 奇幻)",(动画),0.096,0.144,0.052,0.541667,3.761574,0.038176,1.867636,0.812117
25,(动画),"(冒险, 奇幻)",0.144,0.096,0.052,0.361111,3.761574,0.038176,1.414957,0.857656


# 2对演员进行分析

In [9]:
mov=movies[['片名','主演']]
mov.head()

,片名,主演
0,肖申克的救赎,蒂姆·罗宾斯 / 摩根·弗里曼 / 鲍勃·冈顿 / 威廉姆·赛德勒 / 克兰西·布朗 / 吉...
1,霸王别姬,张国荣 / 张丰毅 / 巩俐 / 葛优 / 英达 / 蒋雯丽 / 吴大维 / 吕齐 / 雷汉...
2,阿甘正传,汤姆·汉克斯 / 罗宾·怀特 / 加里·西尼斯 / 麦凯尔泰·威廉逊 / 莎莉·菲尔德 / ...
3,这个杀手不太冷,让·雷诺 / 娜塔莉·波特曼 / 加里·奥德曼 / 丹尼·爱罗 / 彼得·阿佩尔 / 迈克尔...
4,泰坦尼克号,莱昂纳多·迪卡普里奥 / 凯特·温丝莱特 / 比利·赞恩 / 凯西·贝茨 / 弗兰西丝·费舍...


In [10]:
# 第一步，将原始数据转化为标准格式
movies_standard = mov.drop('主演', axis=1).join(mov['主演'].replace(r'\s*/\s*', '|', regex=True).str.strip().str.get_dummies())

In [11]:
#movies_standard = mov.drop('类型', 1).join(mov['类型'].str.get_dummies())
print(movies_standard.head(10))
# 一共包含 9742 部电影，一共有20种不同的电影类型（有2列是ID和电影名）
print(movies_standard.shape)  # (250, 3774)

        片名  A. Cameron Grant  Al Roberts  Alexandra Astin  Alissa Wilms  \
0   肖申克的救赎                 0           0                0             0   
1     霸王别姬                 0           0                0             0   
2     阿甘正传                 0           0                0             0   
3  这个杀手不太冷                 0           0                0             0   
4    泰坦尼克号                 0           0                0             0   
5     美丽人生                 0           0                0             0   
6     千与千寻                 0           0                0             0   
7   辛德勒的名单                 0           0                0             0   
8     盗梦空间                 0           0                0             0   
9  忠犬八公的故事                 0           0                0             0   

   Amber Havens  Amerigo Tot  Angelo Pellegrino  Annton Berry Jr.  \
0             0            0                  0                 0   
1             0            0        

In [12]:
movies_standard.columns

Index(['片名', 'A. Cameron Grant', 'Al Roberts', 'Alexandra Astin',
       'Alissa Wilms', 'Amber Havens', 'Amerigo Tot', 'Angelo Pellegrino',
       'Annton Berry Jr.', 'Antonella Elia',
       ...
       '黛安·克鲁格', '黛安·基顿', '黛安·韦斯特', '黛比·特纳', '黛比·雷诺斯', '齐·麦克布赖德', '齐藤昌',
       '龙田直树', '龚蓓苾', '龟井芳子'],
      dtype='object', length=3774)

In [ ]:
# 利用mlxtend提供的apriori算法函数得到频繁项集，其中设置最小支持度为0.05
movies_standard.set_index(['片名'], inplace=True)

In [14]:
frequent_item_sets = apriori(movies_standard, min_support=0.03, use_colnames=True)
print(frequent_item_sets)

   support itemsets
0    0.032    (张国荣)
1    0.032    (梁朝伟)


D:\soft\python\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:113: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  DeprecationWarning,


In [15]:
# 计算规则，并设置提升度阈值为 1.25 （返回的是各个指标的数值，可以按照按兴趣的指标排序观察，但具体解释还得参考实际数据的含义）
rules = association_rules(frequent_item_sets, metric='lift', min_threshold=1.25)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric


In [16]:
# 对lift降序排序，查看lift较大的是哪些规则
rules_sort = rules.sort_values(by=['lift'], ascending=False)
rules_sort

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
